In [ ]:
import sys
import os
import importlib
import pandas as pd

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s › %(message)s",
    datefmt="%H:%M:%S",
)


# a raiz do projeto (onde fica o notebook) deve estar no path
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Célula 2 – recarregar módulos alterados
# ──────────────────────────────────────────────────────────────────────────────
import importlib

import extract.sheets_fetcher   as sf_mod
import treat.utils.write_back   as wb_mod      # agora separado para evitar circular import
import treat.treat_pipeline     as tp_mod
import treat.treat_runner       as tr_mod

importlib.reload(sf_mod)
importlib.reload(wb_mod)
importlib.reload(tp_mod)
importlib.reload(tr_mod)


In [ ]:
creds_path     = "creds.json"
spreadsheet_id = "1DazUQxspLgT0utOFHcTINbFngXw7Fq0LOq6v4lRGixg"
sheet_name     = "linkedinGeral"

In [ ]:
from extract.sheets_fetcher import SheetsFetcher

fetcher = SheetsFetcher(spreadsheet_id, creds_path)
df_dict = fetcher.get([sheet_name])
df_raw  = df_dict[sheet_name]

print(f"▶️ Dados crus: {df_raw.shape[0]} linhas × {df_raw.shape[1]} colunas")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 15)
df_raw.head(3)

In [ ]:
from treat.treat_pipeline        import TreatPipeline
from treat.utils.renomeacoes     import renomeacao_geral

pipeline = TreatPipeline(
    creds_path         = creds_path,
    spreadsheet_id     = spreadsheet_id,
    sheet_name         = sheet_name,
    mapping_renomeacao = renomeacao_geral,
    write_back         = False,    # mudar para True em produção
)

df_ok = pipeline.run(df_raw)

# mostrar todas as colunas, mas só 5 linhas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)
df_ok.head(10)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Célula X – testar write-back na origem em dry run
# ──────────────────────────────────────────────────────────────────────────────

from load.origin_writer import write_back_origin

# executa em dry_run=True para não tocar no Sheets de verdade
df_wb = write_back_origin(
    df_raw          = df_raw,
    df_ok           = df_ok,
    creds_path      = creds_path,
    spreadsheet_id  = spreadsheet_id,
    sheet_name      = sheet_name,
    write_back      = True,    # liga a lógica de write-back
    dry_run         = False,    # mas evita alteração real
)

print(f"▶️ Dry-run: {df_wb.shape[0]} linhas × {df_wb.shape[1]} colunas preparadas")
df_wb.head(5)


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Celula Y – recarregar dest_writer e importar função
# ──────────────────────────────────────────────────────────────────────────────

import importlib
import load.dest_writer as dw_mod
importlib.reload(dw_mod)

from load.dest_writer import write_back_for_sheet


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Célula Z – testar write-back destino em dry-run
# ──────────────────────────────────────────────────────────────────────────────

# 1) Preparar o DataFrame no formato do modelo
from treat.utils.renomeacoes import renomear_colunas_origem_para_modelo
from treat.utils.campos_calculados import calcular_engajamento_total
from treat.utils.campos_calculados import gerar_id

df_model = renomear_colunas_origem_para_modelo(df_ok, renomeacao_geral)
df_model = calcular_engajamento_total(df_model)
df_model["ID"] = df_model.apply(gerar_id, axis=1)

# 2) Dry-run do write-back na aba de destino
from load.dest_writer import write_back_for_sheet

df_dest = write_back_for_sheet(
    df_model        = df_model,
    sheet_name      = sheet_name,    # ex.: "tiktokAlcance"
    creds_path      = creds_path,
    spreadsheet_id  = spreadsheet_id,
    write_back      = True,          # habilita a rotina
    dry_run         = True,          # evita chamada real ao Sheets
)

print(f"▶️ Dry-run destino: {df_dest.shape[0]} linhas × {df_dest.shape[1]} colunas")
df_dest.head(5)


In [ ]:
import importlib
import treat.bi_param_utils as bp
importlib.reload(bp)

import importlib, utils.preview_links as pl
importlib.reload(pl)


df_ok = pipeline.run(df_raw)
import importlib, treat.treat_pipeline as tp
importlib.reload(tp)

